## Cell 1: setup

In [2]:
import sys
from pathlib import Path
import numpy as np, pandas as pd, rasterio
import matplotlib.pyplot as plt
from PIL import Image

ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT / "backend"))
sys.path.insert(0, str(ROOT / "eval"))

from pipeline import predict_relative_depth
from pipeline.segmentation import remap_dfc_labels, CLASS_NAMES
import metrics as M

RAW = ROOT / "data" / "raw" / "dfc2019"
N_TILES = 20   # raise later for a more reliable estimate

## Cell 2: tile loading

In [3]:
def find_tiles():
    tiles = []
    for rgb in sorted(RAW.rglob("*_RGB.tif")):
        base = rgb.name[:-len("_RGB.tif")]
        agl = next(RAW.rglob(f"{base}_AGL.tif"), None)
        cls = next(RAW.rglob(f"{base}_CLS.tif"), None)
        if agl and cls:
            tiles.append((base, rgb, agl, cls))
    return tiles

def read_rgb(path):
    with rasterio.open(path) as src:
        arr = src.read([1, 2, 3]).transpose(1, 2, 0).astype(np.float32)
    if arr.max() > 255:
        lo, hi = np.percentile(arr, (2, 98))
        arr = np.clip((arr - lo) / (hi - lo + 1e-6), 0, 1) * 255
    return Image.fromarray(arr.astype(np.uint8))

def read_band(path):
    with rasterio.open(path) as src:
        return src.read(1)

tiles = find_tiles()
print(len(tiles), "complete tiles")
idx = np.linspace(0, len(tiles) - 1, N_TILES).astype(int)   # evenly spread, reproducible
sample = [tiles[i] for i in idx]

2783 complete tiles


## Cell 3: run the backbone on the sample

In [ ]:
rows, cache = [], {}
for name, rgb_p, agl_p, cls_p in sample:
    img = read_rgb(rgb_p)
    depth = predict_relative_depth(img)
    agl = read_band(agl_p).astype(np.float32)
    labels = remap_dfc_labels(read_band(cls_p))

    raw = M.compute_metrics(depth, agl)                     # raw corr = direction + shape
    aligned, a, b = M.affine_align(depth, agl, mask=labels != 255)
    glob = M.compute_metrics(aligned, agl, mask=labels != 255)
    cls_corr = {k: v["corr"] for k, v in M.per_class_metrics(depth, agl, labels, CLASS_NAMES).items()}

    rows.append({"tile": name, "raw_corr": raw["corr"], "global_rmse": glob["rmse"],
                 "global_mae": glob["mae"], "scale": a, "shift": b, **{f"corr_{k}": v for k, v in cls_corr.items()}})
    cache[name] = (img, depth, agl)

df = pd.DataFrame(rows)
df